# Experiment health and measurement decisions

Stage 4 read on **`campaign_measurement_decisions`** and **`experiment_health_metrics`**.

This notebook does **not** recompute treatment effects. It shows whether each randomized holdout is structurally trustworthy, how attributed ROAS compares with experimental iROAS, and which deterministic decision the rules assign.

All figures are **synthetic**. They are not real advertiser or retailer results.

**Inputs:** `data/processed/campaign_measurement_decisions.csv` and `data/processed/experiment_health_metrics.csv` from `scripts/run_experiment_decisions.py --export-csv`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
DECISION_CSV = PROCESSED / "campaign_measurement_decisions.csv"
HEALTH_CSV = PROCESSED / "experiment_health_metrics.csv"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}" if pd.notna(x) and abs(x) < 1e6 else f"{x:,.2f}")

dec = pd.read_csv(DECISION_CSV)
health = pd.read_csv(HEALTH_CSV)
print(f"decision rows: {len(dec):,}")
print(f"health rows:   {len(health):,}")

decision rows: 24
health rows:   24


## Which experiments are healthy enough to interpret?

`FAIL` withholds causal interpretation. `WARN` is reviewable and does not automatically invalidate the estimate. `PASS` means the required structural checks succeeded.

In [2]:
print(health["experiment_health_status"].value_counts().to_string())
print()
print("SRM flags:", health["srm_flag"].value_counts().to_dict())
print("Leakage flags:", health["control_exposure_leakage_flag"].value_counts().to_dict())
print("Completeness flags:", health["outcome_completeness_flag"].value_counts().to_dict())
print("Duplicate flags:", health["duplicate_assignment_flag"].value_counts().to_dict())
print("Balance flags:", health["baseline_balance_flag"].value_counts().to_dict())
print()
print(f"max control impressions: {health['control_impressions'].max()}")
print(f"max missing outcomes:    {health['missing_member_outcome_count'].max()}")
print(f"min SRM p-value:         {health['srm_p_value'].min():.4f}")

warn = health.loc[health["experiment_health_status"].eq("WARN"), [
    "campaign_id",
    "experiment_health_status",
    "health_reason",
    "srm_p_value",
    "preperiod_conversion_smd",
    "signup_tenure_smd",
]]
display(warn)

experiment_health_status
PASS    22
WARN     2

SRM flags: {'pass': 24}
Leakage flags: {'pass': 24}
Completeness flags: {'pass': 24}
Duplicate flags: {'pass': 24}
Balance flags: {'pass': 22, 'warn': 2}

max control impressions: 0
max missing outcomes:    0
min SRM p-value:         0.1010


,campaign_id,experiment_health_status,health_reason,srm_p_value,preperiod_conversion_smd,signup_tenure_smd
18,24,WARN,baseline_imbalance,0.6972,-0.0070,0.1029
20,1,WARN,baseline_imbalance,0.9203,-0.0343,0.1126


## Incremental revenue evidence

The primary causal metric is estimated incremental revenue. A campaign is treated as having convincing positive evidence only when the 95% CI lies entirely above zero **and** experiment health is not `FAIL`.

In [3]:
interpretable = dec.loc[dec["experiment_health_status"].ne("FAIL")].copy()
convincing = interpretable.loc[interpretable["incremental_revenue_ci_lower"] > 0]
uncertain = interpretable.loc[
    (interpretable["incremental_revenue_ci_lower"] <= 0)
    & (interpretable["incremental_revenue_ci_upper"] >= 0)
]
print(f"interpretable campaigns: {len(interpretable)}")
print(f"CI entirely positive:    {len(convincing)}")
print(f"CI includes zero:        {len(uncertain)}")

cols = [
    "campaign_id",
    "incremental_revenue",
    "incremental_revenue_ci_lower",
    "incremental_revenue_ci_upper",
    "iroas",
    "experiment_health_status",
    "measurement_decision",
]
display(convincing[cols].sort_values("incremental_revenue", ascending=False).head(8))

interpretable campaigns: 24
CI entirely positive:    17
CI includes zero:        7


,campaign_id,incremental_revenue,incremental_revenue_ci_lower,incremental_revenue_ci_upper,iroas,experiment_health_status,measurement_decision
8,9,"4,626.2400","2,652.7006","6,676.9520",1.3378,PASS,increase_budget
15,16,"4,401.0300","1,739.3488","7,040.8733",1.2565,PASS,increase_budget
3,4,"3,667.6600","1,943.4546","5,166.9516",3.0559,PASS,increase_budget
5,6,"3,404.2100","1,849.4550","4,744.4041",3.2429,PASS,increase_budget
21,22,"3,387.1300",331.2145,"6,082.5135",1.0454,PASS,increase_budget
14,15,"3,337.7900","2,335.1206","4,210.1780",2.5092,PASS,increase_budget
17,18,"3,117.8400",656.7338,"5,591.9086",0.8942,PASS,maintain
20,21,"2,855.2600","1,528.9325","4,215.1907",1.6883,PASS,increase_budget


## Where ROAS and iROAS disagree

Attributed ROAS uses `source_campaign_id` credit. iROAS uses randomized incremental revenue divided by spend. High ROAS with a weak or uncertain iROAS is the project's main commercial lesson.

In [4]:
print(dec["attribution_incrementality_alignment"].value_counts().to_string())
print()
cmp = dec[
    [
        "campaign_id",
        "spend_usd",
        "attributed_revenue_usd",
        "roas",
        "incremental_revenue",
        "iroas",
        "iroas_ci_lower",
        "iroas_ci_upper",
        "attribution_incrementality_alignment",
        "measurement_decision",
    ]
].sort_values("roas", ascending=False)
display(cmp.head(10))

attribution_incrementality_alignment
aligned_positive                            14
inconclusive                                 9
attribution_stronger_than_incrementality     1



,campaign_id,spend_usd,attributed_revenue_usd,roas,incremental_revenue,iroas,iroas_ci_lower,iroas_ci_upper,attribution_incrementality_alignment,measurement_decision
5,6,"1,049.7500","3,337.5400",3.1794,"3,404.2100",3.2429,1.7618,4.5196,aligned_positive,increase_budget
3,4,"1,200.2000","3,497.7000",2.9143,"3,667.6600",3.0559,1.6193,4.3051,aligned_positive,increase_budget
22,23,631.5500,"1,813.1400",2.8709,"1,875.4000",2.9695,0.8257,4.6772,aligned_positive,increase_budget
4,5,963.0500,"2,595.4900",2.6951,"2,745.3500",2.8507,1.5489,4.1200,aligned_positive,increase_budget
20,21,"1,691.2420","2,922.9200",1.7283,"2,855.2600",1.6883,0.9040,2.4924,aligned_positive,increase_budget
8,9,"3,458.2240","5,579.0000",1.6133,"4,626.2400",1.3378,0.7671,1.9307,aligned_positive,increase_budget
6,7,915.4500,"1,446.2100",1.5798,"1,787.4500",1.9525,0.6710,3.0394,aligned_positive,increase_budget
0,1,"1,179.9760","1,837.3800",1.5571,"2,273.1700",1.9265,1.1208,2.7468,aligned_positive,increase_budget
19,20,"1,715.3000","2,408.5900",1.4042,"2,515.6000",1.4666,0.0838,2.7217,aligned_positive,increase_budget
7,8,"1,843.6500","2,379.9200",1.2909,"2,461.1300",1.3349,0.4774,2.1357,aligned_positive,increase_budget


## Health-aware decisions

Rules are deterministic. A failed experiment cannot receive `increase_budget` even if the point estimate is large.

In [5]:
print(dec["measurement_decision"].value_counts().to_string())
print()
display(
    dec[
        [
            "campaign_id",
            "experiment_health_status",
            "absolute_lift",
            "incremental_revenue",
            "iroas",
            "roas",
            "measurement_decision",
            "decision_reason",
        ]
    ].sort_values(["measurement_decision", "campaign_id"])
)

measurement_decision
increase_budget    14
inconclusive        7
maintain            3



,campaign_id,experiment_health_status,absolute_lift,incremental_revenue,iroas,roas,measurement_decision,decision_reason
1,2,PASS,0.0241,"1,792.9500",0.8791,1.0155,inconclusive,positive_incremental_revenue_point_estimate_bu...
2,3,PASS,0.0182,"1,538.3900",1.2287,1.1989,inconclusive,positive_incremental_revenue_point_estimate_bu...
9,10,PASS,0.0108,"1,077.4200",0.9533,0.7064,inconclusive,positive_incremental_revenue_point_estimate_bu...
10,11,PASS,0.0109,"1,150.5000",1.0691,0.8979,inconclusive,positive_incremental_revenue_point_estimate_bu...
11,12,PASS,0.0155,"1,015.8300",0.3464,0.4787,inconclusive,positive_incremental_revenue_point_estimate_bu...
16,17,PASS,0.0135,"1,465.4100",0.6339,0.8150,inconclusive,positive_incremental_revenue_point_estimate_bu...
18,19,PASS,0.0154,"1,738.9300",0.7199,0.8603,inconclusive,positive_incremental_revenue_point_estimate_bu...
0,1,WARN,0.0285,"2,273.1700",1.9265,1.5571,increase_budget,incremental_revenue_ci_positive_and_iroas_meet...
3,4,PASS,0.0431,"3,667.6600",3.0559,2.9143,increase_budget,incremental_revenue_ci_positive_and_iroas_meet...
4,5,PASS,0.0352,"2,745.3500",2.8507,2.6951,increase_budget,incremental_revenue_ci_positive_and_iroas_meet...
